In [1]:
import pandas as pd
import joblib

model = joblib.load("../models/podium_model.pkl")

train = pd.read_parquet("../data_processed/train.parquet")
val = pd.read_parquet("../data_processed/val.parquet")
test = pd.read_parquet("../data_processed/test.parquet")

feature_cols = [
    'feat_grid', 'feat_driver_form_last3', 'feat_team_form_last3',
    'feat_circuit_history', 'feat_driver_dnf_rate_last5',
    'prev_points', 'prev_standing_position'
]

print("Model and data loaded.")

Model and data loaded.


In [2]:
season_2023 = test[test['year'] == 2023].copy()
print(season_2023.shape)
print(season_2023['round'].nunique(), "rounds")

(440, 42)
22 rounds


In [3]:
X_2023 = season_2023[feature_cols]
season_2023['pred_prob'] = model.predict_proba(X_2023)[:, 1]

In [4]:
season_2023['predicted_rank'] = (
    season_2023.groupby('raceId')['pred_prob']
    .rank(method='first', ascending=False)
)

season_2023[['raceId', 'round', 'driverId', 'pred_prob', 'predicted_rank', 'positionOrder']].sort_values(['round', 'predicted_rank']).head(20)

,raceId,round,driverId,pred_prob,predicted_rank,positionOrder
446,1098,1,830,0.944529,1.0,1
443,1098,1,815,0.917306,2.0,2
451,1098,1,844,0.914027,3.0,19
447,1098,1,832,0.818710,4.0,4
453,1098,1,847,0.725105,5.0,7
440,1098,1,1,0.596209,6.0,5
441,1098,1,4,0.552483,7.0,3
448,1098,1,839,0.322901,8.0,18
452,1098,1,846,0.263905,9.0,17
449,1098,1,840,0.216175,10.0,6


In [5]:
season_2023[season_2023['driverId'] == 844][
    ['round', 'feat_grid', 'prev_standing_position', 'prev_points', 'pred_prob', 'positionOrder']
].head(5)

,round,feat_grid,prev_standing_position,prev_points,pred_prob,positionOrder
451,1,3.0,2.0,308.0,0.914027,19
471,2,12.0,19.0,0.0,0.200273,7
491,3,7.0,8.0,6.0,0.461726,20
511,4,1.0,10.0,6.0,0.793328,3
531,5,7.0,6.0,28.0,0.587182,7


In [6]:
# Official F1 points system for top 10
points_system = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

season_2023['simulated_points'] = season_2023['predicted_rank'].map(points_system).fillna(0)

# Sum across the season per driver
predicted_championship = (
    season_2023.groupby('driverId')['simulated_points']
    .sum()
    .sort_values(ascending=False)
)

print(predicted_championship.head(10))

driverId
830    475.0
815    266.0
1      248.0
832    241.0
844    235.0
4      221.0
847    171.0
846    137.0
857     73.0
840     51.0
Name: simulated_points, dtype: float64


In [7]:
drivers = pd.read_csv("../Race data from 1950 to 2026 Race 9/drivers.csv", na_values=['\\N'])

predicted_championship_named = predicted_championship.reset_index().merge(
    drivers[['driverId', 'forename', 'surname']], on='driverId', how='left'
)
predicted_championship_named['driver_name'] = predicted_championship_named['forename'] + ' ' + predicted_championship_named['surname']
print(predicted_championship_named[['driver_name', 'simulated_points']].head(10))

       driver_name  simulated_points
0   Max Verstappen             475.0
1     Sergio Pérez             266.0
2   Lewis Hamilton             248.0
3     Carlos Sainz             241.0
4  Charles Leclerc             235.0
5  Fernando Alonso             221.0
6   George Russell             171.0
7     Lando Norris             137.0
8    Oscar Piastri              73.0
9     Lance Stroll              51.0


In [8]:
driver_standings = pd.read_csv("../Race data from 1950 to 2026 Race 9/driver_standings.csv", na_values=['\\N'])
races = pd.read_csv("../Race data from 1950 to 2026 Race 9/races.csv", na_values=['\\N'])

standings_2023 = driver_standings.merge(races[['raceId', 'year', 'round']], on='raceId')
standings_2023 = standings_2023[standings_2023['year'] == 2023]

final_round = standings_2023['round'].max()
real_final_standings = standings_2023[standings_2023['round'] == final_round].sort_values('points', ascending=False)

real_final_standings_named = real_final_standings.merge(drivers[['driverId', 'forename', 'surname']], on='driverId', how='left')
real_final_standings_named['driver_name'] = real_final_standings_named['forename'] + ' ' + real_final_standings_named['surname']

print(real_final_standings_named[['driver_name', 'points', 'position']].head(10))

       driver_name  points  position
0   Max Verstappen   575.0       1.0
1     Sergio Pérez   285.0       2.0
2   Lewis Hamilton   234.0       3.0
3  Fernando Alonso   206.0       4.0
4  Charles Leclerc   206.0       5.0
5     Lando Norris   205.0       6.0
6     Carlos Sainz   200.0       7.0
7   George Russell   175.0       8.0
8    Oscar Piastri    97.0       9.0
9     Lance Stroll    74.0      10.0
